In [57]:
import pandas as pd

In [58]:
df = pd.read_csv('expenses.csv', thousands=',')
people = ['ben','sherina','keiton','chris','hyeok','mandy','eric', 'harry','haeyung']

df = df.melt(id_vars=[col for col in df.columns if col not in people], var_name='owed_by', value_name='portion')

# Check data types of relevant columns
print(df[['amount', 'portion', 'TOTAL']].dtypes)
df['amount'] = df['amount'].astype(float)

# Ensure numeric types for calculations
df['amount'] = pd.to_numeric(df['amount'], errors='ignore')
df['portion'] = pd.to_numeric(df['portion'], errors='ignore')
df['TOTAL'] = pd.to_numeric(df['TOTAL'], errors='ignore')

#lower case and trim whitespace
df['paid_by'] = df['paid_by'].str.lower().str.strip()
df['owed_by'] = df['owed_by'].str.lower().str.strip()


df = df[(df['portion']!=0) & (df['portion'].notna())]

amount     float64
portion    float64
TOTAL      float64
dtype: object


In [59]:
df['owed_amount'] = df['amount'] * df['portion'] / df['TOTAL']
df

,Item,description,amount,paid_by,TOTAL,owed_by,portion,owed_amount
0,BNB,by /person/nights,2758.52,ben,1.0,ben,0.164948,455.013609
1,Suburu + fuel,by /person/day,582.54,keiton,1.0,ben,0.160377,93.426226
2,Van + fuel,by /person/day,1383.20,ben,1.0,ben,0.160377,221.833962
3,Ramen,NaN,136.00,chris,5.0,ben,1.000000,27.200000
4,Trader Joes,NaN,49.33,hyeok,5.0,ben,1.000000,9.866000
...,...,...,...,...,...,...,...,...
258,Van + fuel,by /person/day,1383.20,ben,1.0,eric,0.047170,65.245283
277,mensho,NaN,279.65,eric,1.0,eric,0.211538,59.156731
282,hotpot ombu,NaN,318.95,chris,1.0,eric,0.148586,47.391397
284,snowbird locker 2,NaN,17.00,chris,7.0,eric,1.000000,2.428571


In [60]:
for person in people:
    print(f"{person} owes: {df[df['owed_by']==person]['owed_amount'].sum():.2f}")


ben owes: 1152.78
sherina owes: 1151.69
keiton owes: 1131.96
chris owes: 1191.45
hyeok owes: 1352.39
mandy owes: 280.02
eric owes: 346.96
harry owes: 313.36
haeyung owes: 344.42


In [61]:
for person in people:
    print(f"{person} paid: {df[df['paid_by']==person]['owed_amount'].sum():.2f}")


ben paid: 5250.78
sherina paid: 463.29
keiton paid: 582.54
chris paid: 554.57
hyeok paid: 49.33
mandy paid: 0.00
eric paid: 364.53
harry paid: 0.00
haeyung paid: 0.00


In [62]:



df1 = df[df['owed_amount']!=0]
df1 = df1[df1['paid_by']!=df1['owed_by']]

mapping = {'sherina': 'ben', 'haeyung': 'keiton', 'mandy':'eric'}
df1['owed_by'] = df1['owed_by'].replace(mapping)
# df1['paid_by'] = df1['paid_by'].replace(mapping)

df1.reset_index(drop = True, inplace=True)
df1.head(3)

,Item,description,amount,paid_by,TOTAL,owed_by,portion,owed_amount
0,Suburu + fuel,by /person/day,582.54,keiton,1.0,ben,0.160377,93.426226
1,Ramen,NaN,136.00,chris,5.0,ben,1.000000,27.200000
2,Trader Joes,NaN,49.33,hyeok,5.0,ben,1.000000,9.866000


In [63]:
from collections import defaultdict
# Step 1: Calculate net balances for each person
balances = defaultdict(float)
for _, row in df1.iterrows():
    balances[row['paid_by']] += row['owed_amount']
    balances[row['owed_by']] -= row['owed_amount']
balances

defaultdict(float,
            {'keiton': -893.8384446942764,
             'ben': 3007.1668402950168,
             'chris': -636.8823022641816,
             'hyeok': -1303.0613461926127,
             'sherina': 402.432,
             'eric': -262.4519017739638,
             'harry': -313.36484536998466})

In [64]:
 # Step 2: Separate into creditors and debtors
creditors = []
debtors = []
for person, balance in balances.items():
    if balance > 0:
        creditors.append((person, balance))
    elif balance < 0:
        debtors.append((person, -balance))
creditors.sort(key=lambda x: x[1], reverse=True)
debtors.sort(key=lambda x: x[1], reverse=True)

In [65]:
# Step 3: Minimize transactions
minimized_transactions = []
while creditors and debtors:
    creditor, credit_amount = creditors.pop()
    debtor, debt_amount = debtors.pop()

    payment = min(credit_amount, debt_amount)
    minimized_transactions.append({'paid_by': creditor, 'owed_by': debtor, 'owed_amount': payment})

    if credit_amount > payment:
        creditors.append((creditor, credit_amount - payment))
        creditors.sort(key=lambda x: x[1], reverse=True)
    if debt_amount > payment:
        debtors.append((debtor, debt_amount - payment))
        debtors.sort(key=lambda x: x[1], reverse=True)


In [66]:
for transaction in minimized_transactions:
    print(transaction['owed_by'], 'pays', round(transaction['owed_amount'],2), 'to', transaction['paid_by'])

eric pays 262.45 to sherina
harry pays 139.98 to sherina
harry pays 173.38 to ben
chris pays 636.88 to ben
keiton pays 893.84 to ben
hyeok pays 1303.06 to ben


In [ ]:

# Generate report to README.md
import datetime

report = "# Expense Splitting Report\n\n"
report += "Google Sheets Link\n"
report += "https://docs.google.com/spreadsheets/d/1sgjZCzSm74SpFO3mT2y9Xk_OrHESxQ3xAgAuiUvRKkQ/edit?gid=1818656043#gid=1818656043\n\n"
report += "owed_amount = amount * portion / total\n\n"
report += f"Report generated on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"


# Total expenses
total_expenses = df['owed_amount'].sum()
report += f"## Total Expenses\n{total_expenses:.2f}\n\n"

# Amount paid and owed
paid_by_person = df.groupby('paid_by')['owed_amount'].sum()
owed_by_person = df.groupby('owed_by')['owed_amount'].sum()

report += "## Summary by Person\n\n"
report += "| Person | Paid | Owes | Net Balance |\n"
report += "|--------|------|------|-------------|\n"
for person in people:
    paid = paid_by_person.get(person, 0)
    owed = owed_by_person.get(person, 0)
    net = paid-owed
    balance_str = f"{net:.2f}" if net != 0 else "0.00"
    report += f"| {person} | {paid:.2f} | {owed:.2f} | {balance_str} |\n"
report += "\n"

report += "## Minimized Transactions\n"

for sugar_baby, sugar_daddy in mapping.items():
    report += f"Sugar daddy {sugar_daddy} pays for sugar baby {sugar_baby}\n"
for transaction in minimized_transactions:
    report += f"- {transaction['owed_by']} pays {round(transaction['owed_amount'],2)} to {transaction['paid_by']}\n"

report += 'Venmo: @benzhong\n'
with open('README.md', 'w') as f:
    f.write(report)
print("Report generated and saved to README.md")

Report generated and saved to README.md
